In [6]:
%cd /content
!rm -rf TrackID3x3
!git clone https://github.com/open-starlab/TrackID3x3.git
%cd /content/TrackID3x3/BoT-SORT

!sed -i '/onnx/Id' requirements.txt
!pip install -q -r requirements.txt
!pip install -q cython
!pip install -q 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip install -q cython_bbox gdown loguru faiss-cpu

path = "/content/TrackID3x3/BoT-SORT/tracker/bot_sort.py"
with open(path, "r") as f:
    content = f.read()
old_line = "from fast_reid.fast_reid_interfece import FastReIDInterface"
if old_line in content:
    new_block = """try:
    from fast_reid.fast_reid_interfece import FastReIDInterface
except Exception as e:
    FastReIDInterface = None
    print("Note: fast_reid unavailable (skipping ReID):", e)"""
    content = content.replace(old_line, new_block)
    with open(path, "w") as f:
        f.write(content)
print("Setup complete.")

/content
Cloning into 'TrackID3x3'...
remote: Enumerating objects: 4123, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 4123 (delta 24), reused 12 (delta 12), pack-reused 4093 (from 1)
Receiving objects: 100% (4123/4123), 289.39 MiB | 27.07 MiB/s, done.
Resolving deltas: 100% (2332/2332), done.
Updating files: 100% (1747/1747), done.
/content/TrackID3x3/BoT-SORT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 17.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [7]:
import os
os.makedirs("pretrained", exist_ok=True)
%cd pretrained
!gdown --id 1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
%cd ..

os.makedirs("videos", exist_ok=True)
!gdown --id 13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M -O videos/outdoor_sample.mp4

/content/TrackID3x3/BoT-SORT/pretrained
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
From (redirected): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5&confirm=t&uuid=c2c00847-42b5-4dda-a97b-b2a4a99f184a
To: /content/TrackID3x3/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar
100% 793M/793M [00:10<00:00, 73.0MB/s]
/content/TrackID3x3/BoT-SORT
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M
From (redirected): https://drive.google.com/uc?id=

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [9]:
!grep -rn "np\.float\b" /content/TrackID3x3/BoT-SORT --include="*.py"

/content/TrackID3x3/BoT-SORT/tracker/mc_bot_sort.py:20:        self._tlwh = np.asarray(tlwh, dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/bot_sort.py:24:        self._tlwh = np.asarray(tlwh, dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/matching.py:60:    ious = np.zeros((len(atlbrs), len(btlbrs)), dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/matching.py:65:        np.ascontiguousarray(atlbrs, dtype=np.float),
/content/TrackID3x3/BoT-SORT/tracker/matching.py:66:        np.ascontiguousarray(btlbrs, dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/matching.py:136:    cost_matrix = np.zeros((len(tracks), len(detections)), dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/matching.py:139:    det_features = np.asarray([track.curr_feat for track in detections], dtype=np.float)
/content/TrackID3x3/BoT-SORT/tracker/matching.py:140:    track_features = np.asarray([track.smooth_feat for track in tracks], dtype=np.float)


In [10]:
import subprocess

result = subprocess.run(
    ["grep", "-rl", "np.float\\b", "/content/TrackID3x3/BoT-SORT", "--include=*.py"],
    capture_output=True, text=True
)
files = result.stdout.strip().split("\n")
files = [f for f in files if f]

for filepath in files:
    with open(filepath, "r") as f:
        content = f.read()
    # np.float -> float (but not np.float32/64 etc, which are still valid)
    import re
    new_content = re.sub(r"np\.float\b(?!\d)", "float", content)
    if new_content != content:
        with open(filepath, "w") as f:
            f.write(new_content)
        print("Patched:", filepath)

print("Done.")

Patched: /content/TrackID3x3/BoT-SORT/tracker/mc_bot_sort.py
Patched: /content/TrackID3x3/BoT-SORT/tracker/bot_sort.py
Patched: /content/TrackID3x3/BoT-SORT/tracker/matching.py
Done.


In [11]:
!sed -n '270,285p' tools/demo.py

                online_im = plot_tracking(
                    img_info['raw_img'], online_tlwhs, online_ids, frame_id=frame_id + 1, fps=1. / timer.average_time
                )
            else:
                timer.toc()
                online_im = img_info['raw_img']
            if args.save_result:
                vid_writer.write(online_im)
            ch = cv2.waitKey(1)
            if ch == 27 or ch == ord("q") or ch == ord("Q"):
                break
        else:
            break
        frame_id += 1

    if args.save_result:


In [12]:
%cd /content/TrackID3x3/BoT-SORT

# Show line 278 to confirm current content
!sed -n '278p' tools/demo.py

/content/TrackID3x3/BoT-SORT
            ch = cv2.waitKey(1)


In [13]:
!sed -i '278s/.*/            ch = 0  # headless Colab: no display/' tools/demo.py

# Confirm the change took effect
!sed -n '278p' tools/demo.py

            ch = 0  # headless Colab: no display


In [14]:
!pip install -q ffmpeg-python
!ffmpeg -y -i videos/outdoor_sample.mp4 -t 10 -c copy videos/outdoor_short.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [15]:
!pwd
!ls

/content/TrackID3x3/BoT-SORT
assets	   pretrained	     setup.cfg	tracker		       yolov7
fast_reid  README.md	     setup.py	VideoCameraCorrection  yolox
LICENSE    requirements.txt  tools	videos


In [16]:
%cd /content/TrackID3x3/BoT-SORT
!ls tools/demo.py

/content/TrackID3x3/BoT-SORT
tools/demo.py


In [17]:
!grep -n "frame_id % 20" tools/demo.py

204:        if frame_id % 20 == 0:
239:        if frame_id % 20 == 0:


In [18]:
!sed -i 's/frame_id % 20/frame_id % 1/' tools/demo.py

In [19]:
!ffmpeg -y -i videos/outdoor_short.mp4 -vf scale=640:360 -c:v libx264 videos/outdoor_tiny.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [20]:
!python3 tools/demo.py video \
    -f yolox/exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --path videos/outdoor_tiny.mp4 \
    --fp16 --fuse --save_result --device gpu


Note: fast_reid unavailable (skipping ReID): cannot import name 'Mapping' from 'collections' (/usr/lib/python3.13/collections/__init__.py)
2026-10-08 01:52:09.316 | INFO     | __main__:main:307 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/outdoor_tiny.mp4', camid=0, save_result=True, exp_file='yolox/exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, fuse_score=False, cmc_method='orb', with_reid=False, fast_reid_config='fast_reid/configs/MOT17/sbs_S50.yml', fast_reid_weights='pretrained/mot17_sbs_S50.pth', proximity_thresh=0.5, appearance_thresh=0.25, ablation=False, mot20=True)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgri

In [21]:
import glob
output_paths = glob.glob("YOLOX_outputs/**/*.mp4", recursive=True)
print(output_paths)

['YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4']


In [22]:
import glob
output_paths = glob.glob("YOLOX_outputs/**/*.mp4", recursive=True)
print(output_paths)

import os
for p in output_paths:
    print(p, "-", os.path.getsize(p), "bytes")

['YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4']
YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4 - 4035777 bytes


In [23]:
input_path = output_paths[0]  # adjust index if needed
output_path = "/content/playable_output.mp4"

!ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p "{output_path}"

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [24]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(output_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>')

In [25]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
import glob, os

output_paths = glob.glob("YOLOX_outputs/**/*.mp4", recursive=True)

for p in sorted(output_paths, key=os.path.getmtime):
    size_mb = os.path.getsize(p) / (1024*1024)
    mtime = os.path.getmtime(p)
    import datetime
    mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%H:%M:%S")
    print(f"{mtime_str} | {size_mb:.2f} MB | {p}")

01:53:17 | 3.85 MB | YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4


In [27]:
import glob, os

output_paths = glob.glob("YOLOX_outputs/**/*.mp4", recursive=True)

if not output_paths:
    print("No mp4 files found — checking the folder structure instead:")
    !find YOLOX_outputs -type f
else:
    for p in sorted(output_paths, key=os.path.getmtime):
        size_mb = os.path.getsize(p) / (1024*1024)
        print(f"{size_mb:.2f} MB | {repr(p)}")

3.85 MB | 'YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4'


In [28]:
!pwd

/content/TrackID3x3/BoT-SORT


In [29]:
input_path = "YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_26_22_35_43/outdoor_tiny.mp4"

!ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p /content/playable_output.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [30]:
import cv2

cap = cv2.VideoCapture("videos/outdoor_tiny.mp4")
ret, frame = cap.read()
cap.release()

cv2.imwrite("/content/sample_frame.jpg", frame)
print("Frame size:", frame.shape)  # (height, width, channels)

Frame size: (360, 640, 3)


In [31]:
!grep -n "tracker.update" tools/demo.py

172:            online_targets = tracker.update(detections, img_info['raw_img'])
253:                online_targets = tracker.update(detections, img_info["raw_img"])


In [32]:
import re

path = "/content/TrackID3x3/BoT-SORT/tools/demo.py"
with open(path, "r") as f:
    content = f.read()

old_line = 'online_targets = tracker.update(detections, img_info["raw_img"])'

new_block = '''# Filter out detections to the right of x=550 (outside court)
if detections is not None and len(detections) > 0:
    x_centers = (detections[:, 0] + detections[:, 2]) / 2
    detections = detections[x_centers <= 550]

online_targets = tracker.update(detections, img_info["raw_img"])'''

assert old_line in content, "Exact line not found — paste the grep output so I can match it precisely"
content = content.replace(old_line, new_block)

with open(path, "w") as f:
    f.write(content)

print("Patched: detections now filtered to x <= 550")

Patched: detections now filtered to x <= 550


In [33]:
!sed -n '245,265p' tools/demo.py

            scale = min(exp.test_size[0] / float(img_info['height'], ), exp.test_size[1] / float(img_info['width']))

            if outputs[0] is not None:
                outputs = outputs[0].cpu().numpy()
                detections = outputs[:, :7]
                detections[:, :4] /= scale

                # Run tracker
                # Filter out detections to the right of x=550 (outside court)
if detections is not None and len(detections) > 0:
    x_centers = (detections[:, 0] + detections[:, 2]) / 2
    detections = detections[x_centers <= 550]

online_targets = tracker.update(detections, img_info["raw_img"])

                online_tlwhs = []
                online_ids = []
                online_scores = []
                for t in online_targets:
                    tlwh = t.tlwh
                    tid = t.track_id


In [34]:
!git diff tools/demo.py

diff --git a/BoT-SORT/tools/demo.py b/BoT-SORT/tools/demo.py
index b2c0bd2..46c8a4e 100755
--- a/BoT-SORT/tools/demo.py
+++ b/BoT-SORT/tools/demo.py
@@ -201,7 +201,7 @@ def image_demo(predictor, vis_folder, current_time, args):
             os.makedirs(save_folder, exist_ok=True)
             cv2.imwrite(osp.join(save_folder, osp.basename(img_path)), online_im)
 
-        if frame_id % 20 == 0:
+        if frame_id % 1 == 0:
             logger.info('Processing frame {} ({:.2f} fps)'.format(frame_id, 1. / max(1e-5, timer.average_time)))
 
         ch = cv2.waitKey(0)
@@ -236,7 +236,7 @@ def imageflow_demo(predictor, vis_folder, current_time, args):
     frame_id = 0
     results = []
     while True:
-        if frame_id % 20 == 0:
+        if frame_id % 1 == 0:
             logger.info('Processing frame {} ({:.2f} fps)'.format(frame_id, 1. / max(1e-5, timer.average_time)))
         ret_val, frame = cap.read()
         if ret_val:
@@ -250,7 +250,12 @@ def imageflow_demo(predictor, vis_

In [35]:
!git checkout -- tools/demo.py

In [36]:
!grep -n "Filter out detections\|online_targets = tracker.update\|x_centers" tools/demo.py

172:            online_targets = tracker.update(detections, img_info['raw_img'])
253:                online_targets = tracker.update(detections, img_info["raw_img"])


In [37]:
!sed -n '245,260p' tools/demo.py

            scale = min(exp.test_size[0] / float(img_info['height'], ), exp.test_size[1] / float(img_info['width']))

            if outputs[0] is not None:
                outputs = outputs[0].cpu().numpy()
                detections = outputs[:, :7]
                detections[:, :4] /= scale

                # Run tracker
                online_targets = tracker.update(detections, img_info["raw_img"])

                online_tlwhs = []
                online_ids = []
                online_scores = []
                for t in online_targets:
                    tlwh = t.tlwh
                    tid = t.track_id


In [38]:
path = "tools/demo.py"

with open(path, "r") as f:
    lines = f.readlines()

target_line_num = 253
target_index = target_line_num - 1

assert "online_targets = tracker.update" in lines[target_index], f"Line {target_line_num} doesn't match expected content: {lines[target_index]}"

indent = "                "  # 16 spaces

filter_lines = [
    f"{indent}# Filter out detections to the right of x=550 (outside court)\n",
    f"{indent}if detections is not None and len(detections) > 0:\n",
    f"{indent}    x_centers = (detections[:, 0] + detections[:, 2]) / 2\n",
    f"{indent}    detections = detections[x_centers <= 550]\n",
    f"{indent}\n",
]

new_lines = lines[:target_index] + filter_lines + lines[target_index:]

with open(path, "w") as f:
    f.writelines(new_lines)

print("Patched successfully at line", target_line_num)

Patched successfully at line 253


In [39]:
!sed -n '245,265p' tools/demo.py

            scale = min(exp.test_size[0] / float(img_info['height'], ), exp.test_size[1] / float(img_info['width']))

            if outputs[0] is not None:
                outputs = outputs[0].cpu().numpy()
                detections = outputs[:, :7]
                detections[:, :4] /= scale

                # Run tracker
                # Filter out detections to the right of x=550 (outside court)
                if detections is not None and len(detections) > 0:
                    x_centers = (detections[:, 0] + detections[:, 2]) / 2
                    detections = detections[x_centers <= 550]
                
                online_targets = tracker.update(detections, img_info["raw_img"])

                online_tlwhs = []
                online_ids = []
                online_scores = []
                for t in online_targets:
                    tlwh = t.tlwh
                    tid = t.track_id


In [40]:
!sed -n '283p' tools/demo.py

            ch = cv2.waitKey(1)


In [41]:
!sed -i '283s/.*/            ch = 0  # headless Colab: no display/' tools/demo.py
!sed -n '283p' tools/demo.py

            ch = 0  # headless Colab: no display


In [42]:
!grep -n "Filter out detections\|onnx\|np.float\|frame_id % 1" tools/demo.py requirements.txt 2>/dev/null

tools/demo.py:253:                # Filter out detections to the right of x=550 (outside court)


In [43]:
!python3 tools/demo.py video \
    -f yolox/exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --path videos/outdoor_tiny.mp4 \
    --fp16 --fuse --save_result --device gpu

Note: fast_reid unavailable (skipping ReID): cannot import name 'Mapping' from 'collections' (/usr/lib/python3.13/collections/__init__.py)
2026-10-08 01:53:37.758 | INFO     | __main__:main:312 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/outdoor_tiny.mp4', camid=0, save_result=True, exp_file='yolox/exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, fuse_score=False, cmc_method='orb', with_reid=False, fast_reid_config='fast_reid/configs/MOT17/sbs_S50.yml', fast_reid_weights='pretrained/mot17_sbs_S50.pth', proximity_thresh=0.5, appearance_thresh=0.25, ablation=False, mot20=True)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgri

In [44]:
import glob, os

output_paths = glob.glob("YOLOX_outputs/**/*.mp4", recursive=True)
for p in sorted(output_paths, key=os.path.getmtime):
    size_mb = os.path.getsize(p) / (1024*1024)
    print(f"{size_mb:.2f} MB | {repr(p)}")

3.85 MB | 'YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_52_14/outdoor_tiny.mp4'
3.71 MB | 'YOLOX_outputs/yolox_x_mix_det/track_vis/2026_10_08_01_53_41/outdoor_tiny.mp4'


In [45]:
input_path = "YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_26_22_29_32/outdoor_sample.mp4"

!ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p /content/playable_output.mp4

from IPython.display import HTML
from base64 import b64encode

mp4 = open("/content/playable_output.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>')

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [46]:
from google.colab import files
files.download("/content/playable_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>